# Silver Layer: Parquet to Delta Conversion

Converts Silver Parquet files to Delta tables for significantly faster investigation queries.

**Benefits:**
- Z-ordering for fast lookups by event_name, principal_id, ip_address
- Data skipping with column statistics
- Partition pruning on ingest_date
- Cached metadata in Unity Catalog

In [ ]:
"""
STEP 1: Convert Silver Audit Logs to Delta Table
"""

from pyspark.sql import functions as F

# ============================================================================
# CONFIGURATION
# ============================================================================

CATALOG = "gitrepo"
SCHEMA = "default"

# Source Parquet paths
SILVER_AUDIT_PARQUET = "/Volumes/gitrepo/default/git_oci_aidp_silver/audit_logs/data"
SILVER_FLOW_PARQUET = "/Volumes/gitrepo/default/git_oci_aidp_silver/flow_logs/data"

# Target Delta tables
SILVER_AUDIT_TABLE = f"{CATALOG}.{SCHEMA}.silver_audit_logs"
SILVER_FLOW_TABLE = f"{CATALOG}.{SCHEMA}.silver_flow_logs"

# Spark tuning for large data
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

print("=" * 70)
print("SILVER PARQUET TO DELTA CONVERSION")
print("=" * 70)
print(f"Audit Logs: {SILVER_AUDIT_PARQUET} -> {SILVER_AUDIT_TABLE}")
print(f"Flow Logs:  {SILVER_FLOW_PARQUET} -> {SILVER_FLOW_TABLE}")
print("=" * 70)

In [ ]:
"""
STEP 2: Convert Audit Logs (17.4M records)
Estimated time: 5-10 minutes
"""

print("\n" + "=" * 70)
print("CONVERTING AUDIT LOGS TO DELTA")
print("=" * 70)

print("\n[1/4] Reading Silver Parquet...")
silver_audit_df = spark.read.parquet(SILVER_AUDIT_PARQUET)
record_count = silver_audit_df.count()
print(f"✓ Loaded {record_count:,} records")

print("\n[2/4] Writing to Delta table...")
(
    silver_audit_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ingest_date")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_AUDIT_TABLE)
)
print(f"✓ Written to {SILVER_AUDIT_TABLE}")

print("\n[3/4] Optimizing with Z-ORDER...")
print("This enables fast lookups by event_name, principal_id, ip_address")
spark.sql(f"""
    OPTIMIZE {SILVER_AUDIT_TABLE}
    ZORDER BY (event_name, principal_id, ip_address)
""")
print("✓ Z-ORDER optimization complete")

print("\n[4/4] Computing statistics...")
spark.sql(f"ANALYZE TABLE {SILVER_AUDIT_TABLE} COMPUTE STATISTICS FOR ALL COLUMNS")
print("✓ Statistics computed")

print("\n" + "=" * 70)
print(f"✓ AUDIT LOGS CONVERSION COMPLETE: {record_count:,} records")
print("=" * 70)

In [ ]:
"""
STEP 3: Convert Flow Logs (161M+ records)
Estimated time: 15-30 minutes (larger dataset)
"""

print("\n" + "=" * 70)
print("CONVERTING FLOW LOGS TO DELTA")
print("=" * 70)

print("\n[1/4] Reading Silver Parquet...")
silver_flow_df = spark.read.parquet(SILVER_FLOW_PARQUET)
flow_count = silver_flow_df.count()
print(f"✓ Loaded {flow_count:,} records")

print("\n[2/4] Writing to Delta table...")
print("This may take 10-20 minutes for 160M+ records...")
(
    silver_flow_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ingest_date")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_FLOW_TABLE)
)
print(f"✓ Written to {SILVER_FLOW_TABLE}")

print("\n[3/4] Optimizing with Z-ORDER...")
print("This enables fast lookups by src_ip, dst_ip, dst_port, action")
spark.sql(f"""
    OPTIMIZE {SILVER_FLOW_TABLE}
    ZORDER BY (src_ip, dst_ip, dst_port, action)
""")
print("✓ Z-ORDER optimization complete")

print("\n[4/4] Computing statistics...")
spark.sql(f"ANALYZE TABLE {SILVER_FLOW_TABLE} COMPUTE STATISTICS FOR ALL COLUMNS")
print("✓ Statistics computed")

print("\n" + "=" * 70)
print(f"✓ FLOW LOGS CONVERSION COMPLETE: {flow_count:,} records")
print("=" * 70)

In [ ]:
"""
STEP 4: Verification & Table Info
"""

print("\n" + "=" * 70)
print("VERIFICATION")
print("=" * 70)

# Audit logs
print("\n--- Silver Audit Logs Delta Table ---")
audit_delta = spark.table(SILVER_AUDIT_TABLE)
print(f"Record count: {audit_delta.count():,}")
print(f"Partitions: {audit_delta.select('ingest_date').distinct().count()} dates")

# Flow logs
print("\n--- Silver Flow Logs Delta Table ---")
flow_delta = spark.table(SILVER_FLOW_TABLE)
print(f"Record count: {flow_delta.count():,}")
print(f"Partitions: {flow_delta.select('ingest_date').distinct().count()} dates")

# Show table details
print("\n--- Table Details ---")
spark.sql(f"DESCRIBE EXTENDED {SILVER_AUDIT_TABLE}").show(50, truncate=False)

print("\n" + "=" * 70)
print("✓ ALL CONVERSIONS COMPLETE")
print("=" * 70)
print("\nYour Delta tables are ready:")
print(f"  • {SILVER_AUDIT_TABLE}")
print(f"  • {SILVER_FLOW_TABLE}")
print("\nUpdate your investigation queries to use these tables!")
print("=" * 70)

## Updated Investigation Queries

After conversion, update your `Investigate_Queries.ipynb` to use Delta tables instead of Parquet:

```python
# OLD (slow - Parquet)
silver_audit = spark.read.parquet(SILVER_AUDIT_PATH).filter(col("ingest_date") == PARTITION_DATE)

# NEW (fast - Delta with partition pruning + Z-ORDER)
silver_audit = spark.table("gitrepo.default.silver_audit_logs").filter(col("ingest_date") == PARTITION_DATE)
```

In [ ]:
"""
STEP 5: Test Query Performance (Optional)
Compare Delta vs Parquet query times
"""

import time

EVENT_NAME = "CreatePrivateIp"
PARTITION_DATE = "2026-01-28"

print("\n" + "=" * 70)
print("PERFORMANCE COMPARISON")
print("=" * 70)

# Test 1: Parquet query
print(f"\n--- Parquet Query: {EVENT_NAME} on {PARTITION_DATE} ---")
start = time.time()
parquet_count = (
    spark.read.parquet(SILVER_AUDIT_PARQUET)
    .filter(F.col("ingest_date") == PARTITION_DATE)
    .filter(F.col("event_name") == EVENT_NAME)
    .count()
)
parquet_time = time.time() - start
print(f"Result: {parquet_count:,} records")
print(f"Time: {parquet_time:.2f} seconds")

# Test 2: Delta query
print(f"\n--- Delta Query: {EVENT_NAME} on {PARTITION_DATE} ---")
start = time.time()
delta_count = (
    spark.table(SILVER_AUDIT_TABLE)
    .filter(F.col("ingest_date") == PARTITION_DATE)
    .filter(F.col("event_name") == EVENT_NAME)
    .count()
)
delta_time = time.time() - start
print(f"Result: {delta_count:,} records")
print(f"Time: {delta_time:.2f} seconds")

# Comparison
print("\n" + "=" * 70)
print("RESULTS")
print("=" * 70)
if parquet_time > 0:
    speedup = parquet_time / delta_time if delta_time > 0 else float('inf')
    print(f"Parquet: {parquet_time:.2f}s")
    print(f"Delta:   {delta_time:.2f}s")
    print(f"Speedup: {speedup:.1f}x faster with Delta")
print("=" * 70)

In [ ]:
%sql
-- Quick test: Query the new Delta table directly with SQL
SELECT 
    event_name,
    COUNT(*) as count,
    COUNT(DISTINCT principal_id) as unique_principals,
    COUNT(DISTINCT ip_address) as unique_ips
FROM gitrepo.default.silver_audit_logs
WHERE ingest_date = '2026-01-28'
GROUP BY event_name
ORDER BY count DESC
LIMIT 20